# PRAGMA Pre-training

This notebook demonstrates the self-supervised pre-training loop for the PRAGMA model. We use Masked Event Prediction, where a percentage of transactions in a sequence are masked, and the model must reconstruct their features.

In [ ]:
import torch
import torch.nn as nn
import sys
sys.path.append('../src')

from pragma_model import PRAGMA

# Real configuration matching IEEEDataset features
profile_config = {
    'num_numerical_features': 5, # C1 to C5
    'num_categorical_features': 3, # card4, card6, P_emaildomain
    'cat_embedding_dims': [(100, 8), (100, 8), (100, 8)] # Hashed mod 100 with dim 8
}

event_config = {
    'event_dim': 20, # TransactionAmt + V1-V19
    'embed_dim': 64,
    'num_heads': 4,
    'num_layers': 3,
    'max_seq_len': 100
}

model = PRAGMA(profile_config, event_config, embed_dim=64)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


In [ ]:
# Optimization
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss() # Using MSE for continuous event reconstruction

## Pre-training Loop

In [ ]:
from torch.utils.data import DataLoader
import sys
sys.path.append('../src')
from data.dataset import IEEEDataset

def pretrain_step(model, optimizer, criterion, batch, mask_prob=0.15):
    model.train()
    
    x_num, x_cat, events, seq_lengths = [b.to(device) for b in batch]
    
    # Create Masked Events (copy original events for labels)
    labels = events.clone()
    
    # Generate mask
    mask = torch.rand(events.shape[:2], device=events.device) < mask_prob
    
    # Don't mask padding tokens
    pad_mask = torch.arange(events.shape[1], device=events.device)[None, :] >= seq_lengths[:, None]
    mask = mask & ~pad_mask
    
    # Apply mask (e.g., zero out masked events)
    masked_events = events.clone()
    masked_events[mask] = 0.0
    
    # Forward pass
    optimizer.zero_grad()
    fused, mlm_preds = model(x_num, x_cat, masked_events, seq_lengths, pretrain=True)
    
    # Compute loss only on masked positions
    loss = criterion(mlm_preds[mask], labels[mask])
    
    loss.backward()
    optimizer.step()
    
    return loss.item()

# Instantiate the real IEEE Dataset
print("Loading dataset...")
# Note: In Colab, you would mount Google Drive or Kaggle API to get the parquet files.
# For this example, we assume make_dataset.py generated train.parquet
try:
    dataset = IEEEDataset('../data/processed/train.parquet', max_seq_len=100)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
    
    # Run a few epochs
    num_epochs = 2
    print("Starting Pre-training Loop...")
    for epoch in range(num_epochs):
        total_loss = 0
        for batch_idx, batch in enumerate(dataloader):
            loss = pretrain_step(model, optimizer, criterion, batch)
            total_loss += loss
            
            if batch_idx % 10 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}], Step [{batch_idx}], Loss: {loss:.4f}")
            
            # Break early for demonstration purposes
            if batch_idx >= 50:
                break
        print(f"Epoch {epoch+1} Average Loss: {total_loss/(batch_idx+1):.4f}")
except FileNotFoundError:
    print("Train.parquet not found. Please run make_dataset.py first.")


In [ ]:
# Save pre-trained checkpoint
torch.save(model.state_dict(), '../models/pragma_pretrained.pth')
print("Pre-trained model saved to ../models/pragma_pretrained.pth")